# Análise Textual de Contratos PNCP

Identificação de **subenquadramento**: contratos rotulados como "serviços gerais"
que na verdade são engenharia/obras (Lei 14.133/2021).

## Roteiro

1. Bibliotecas
2. Métodos de apoio
3. Modelo LLM (Google Gemini)
4. Base textual (contratos PNCP)
5. TF-IDF
6. N-gramas (bigramas e trigramas)
7. Rede k-NN sobre TF-IDF + clusters
8. Embeddings contextuais (SBERT) + clusters
9. Indicadores por cluster (LLM)
10. Classificação semissupervisionada (regularização em grafo)
11. **Detecção de subenquadramento usando os rótulos limpos — FOCO PRINCIPAL**
12. **Análise de rito de engenharia (PDFs anexados — CREA/ART)**
13. Agente LLM com ferramentas
14. Exportação consolidada (.md + CSVs + gráficos)

**Hipótese:** os rótulos `engenharia` e `obras` são confiáveis; o `geral` é
ruidoso (esconde engenharia mal classificada). Usamos os rótulos limpos como
referência para achar os `geral` que deveriam ser engenharia/obras.

**Pré-requisito:** ter `contratos.parquet` no Google Drive (coleta anterior).
Os resultados deste notebook ficam em `resultados_analise_textual/` —
separados dos resultados do pipeline.

## 1. Bibliotecas

In [ ]:
!pip install -q sentence-transformers networkx plotly nltk
!pip install -q google-generativeai
!pip install -q pyarrow pandas

In [ ]:
import re
import time
import json
import unicodedata
import pandas as pd
import numpy as np

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics.pairwise import cosine_similarity

import networkx as nx
from networkx.algorithms import community

import plotly.graph_objects as go


# ── Limpeza de boilerplate de cabeçalho dos objetos PNCP ────────────────────
# Sem isso, o SBERT é dominado por 50 palavras de prefácio idêntico que
# aparecem em quase todo contrato ("contratação de empresa especializada
# para prestação de serviços de..."), o que aproxima TODOS os contratos
# e arruina a discriminação entre engenharia e geral.
_RX_BOILERPLATE = [
    re.compile(p, flags=re.IGNORECASE) for p in [
        r"^\s*registro\s+de\s+pre[çc]os?\s+(para\s+a?\s*)?",
        r"^\s*contrata[çc][ãa]o\s+(de\s+)?(empresa\s+)?(especializada\s+)?"
        r"(em\s+|para\s+(o?\s+|a?\s+)?(presta[çc][ãa]o\s+(de\s+)?)?(servi[çc]os?\s+(de\s+)?)?)?",
        r"^\s*contrata[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?",
        r"^\s*presta[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?",
        r"^\s*aquisi[çc][ãa]o\s+de\s+",
        r"^\s*fornecimento\s+de\s+",
        r"^\s*execu[çc][ãa]o\s+de\s+(servi[çc]os?\s+(de\s+)?)?",
        r"^\s*loca[çc][ãa]o\s+de\s+",
    ]
]
_RX_ESPACO = re.compile(r"\s+")

def limpar_boilerplate(texto):
    """Remove prefixos burocráticos comuns para o SBERT ver o objeto real."""
    if not isinstance(texto, str):
        return ''
    t = texto.strip()
    # Aplica em cascata — vários prefixos se aninham
    for _ in range(3):
        for rx in _RX_BOILERPLATE:
            t = rx.sub('', t, count=1)
    return _RX_ESPACO.sub(' ', t).strip()


# Exemplos:
for ex in ['CONTRATACAO DE EMPRESA ESPECIALIZADA PARA PRESTACAO DE SERVICOS DE LOCUCAO EM EVENTOS',
            'REGISTRO DE PREÇOS PARA A CONTRATAÇÃO DE SERVIÇOS DE ALINHAMENTO DE PNEUS',
            'CONTRATAÇÃO DE SERVIÇOS DE ENGENHARIA PARA CONSTRUÇÃO DE UBS']:
    print(f'antes: {ex}')
    print(f'depois: {limpar_boilerplate(ex)}\n')

## 2. Métodos de apoio

- `meu_tokenizador`: minúsculas + tokenização + stopwords PT-BR + stemmer RSLP.
- `get_cluster_descriptors`: termos com maior soma TF-IDF de um cluster.
- `show_graph` / `show_graph_regularization`: visualização limpa dos grafos —
  **forma por rótulo** (geral = triângulo, engenharia/obras = círculo), cor por
  rótulo ou cluster, tamanho por grau.
- `simple_regularization`: propagação de rótulos no grafo (semissupervisionado).

In [ ]:
STOP_PT = set(nltk.corpus.stopwords.words('portuguese'))

# Stopwords de domínio PNCP — termos burocráticos que aparecem em quase
# todo contrato e poluem TF-IDF e descritores de cluster. Sem isto, os top
# termos viram "contratacao, servicos, empresa, especializada" — inútil
# para discriminar engenharia × geral.
STOP_DOMINIO = {
    'contratacao', 'contratacoes', 'contratar', 'contrato', 'contratos',
    'servico', 'servicos', 'prestacao', 'prestar', 'prestados', 'prestada',
    'empresa', 'empresas', 'especializada', 'especializado',
    'fornecimento', 'fornecer', 'aquisicao', 'aquisicoes', 'fornecedor',
    'registro', 'preco', 'precos',
    'atender', 'atendimento', 'atende', 'atendidas',
    'executar', 'execucao', 'realizacao', 'realizar',
    'objeto', 'objetos', 'objetivando', 'objetivo',
    'conforme', 'demanda', 'demandas', 'referente',
    'pregao', 'licitacao', 'edital', 'editais', 'ata', 'atas',
    'municipio', 'municipal', 'estadual', 'federal',
    'lote', 'lotes', 'item', 'itens',
}
STOP_TUDO = STOP_PT | STOP_DOMINIO
STEMMER = RSLPStemmer()

def remove_stopwords(texto, stop_words=STOP_TUDO):
    s = str(texto).lower()
    # Normaliza acentos para casar com stopwords sem acento
    s_norm = ''.join(c for c in unicodedata.normalize('NFKD', s)
                      if not unicodedata.combining(c))
    tokens = word_tokenize(s_norm, language='portuguese')
    return [w for w in tokens
            if w not in stop_words and w.isalnum() and not w.isdigit()
            and len(w) > 2]

def stemming(tokens, stemmer=STEMMER):
    return [stemmer.stem(w) for w in tokens]

def meu_tokenizador(doc):
    return stemming(remove_stopwords(doc))

def get_cluster_descriptors(VSM, df_documentos, cluster_id, max_terms=5):
    sub = df_documentos[df_documentos.cluster == cluster_id]['text']
    df_desc = pd.DataFrame({
        'word': VSM.get_feature_names_out(),
        'tfidf_sum': VSM.transform(sub).toarray().sum(axis=0),
    }).sort_values('tfidf_sum', ascending=False)
    return len(sub), df_desc[df_desc.tfidf_sum > 0].head(max_terms).word.tolist()


# ── Regex de ENGENHARIA ÓBVIA (sinal local, independente do LLM) ────────────
# Padrões com alta precisão: se o objeto bate com algum, é praticamente
# certo ser engenharia/obras. Funciona como 4º sinal no ranking e como
# fallback quando o LLM está com rate limit.
PADROES_OBVIO_ENG = [re.compile(p, flags=re.IGNORECASE) for p in [
    r"\bconstruc[aã]o\s+(de|do|da|dos|das)\s+"
    r"(predio|edif[ií]cio|escola|creche|posto|ubs|upa|hospital|"
    r"unidade\s+de|centro\s+de|ginas[ií]o|quadra|ponte|viaduto|"
    r"passarela|bueiro|muro|cerca|reservat[oó]rio|po[cç]o|casa|"
    r"estrutura)\b",
    r"\breforma\s+(estrutural|predial|geral|completa|de\s+coberta|"
    r"com\s+amplia[cç][aã]o|do\s+(predio|edif[ií]cio|hospital|escola))\b",
    r"\bampliac[aã]o\s+(de|do|da)\s+"
    r"(predio|escola|hospital|ponte|edif[ií]cio|unidade)\b",
    r"\bpavimentac[aã]o\s+(asf[aá]ltica|com\s+asfalto|em\s+(cbuq|cbq)|"
    r"de\s+via|de\s+rua|paralelep[ií]pedo)\b",
    r"\brecapeamento\s+asf[aá]ltico\b",
    r"\bdrenagem\s+(pluvial|urbana|de\s+via|de\s+[aá]guas)\b",
    r"\bterraplenagem\b",
    r"\bobra\s+(civil|de\s+arte\s+especial|de\s+engenharia)\b",
    r"\brede\s+de\s+(esgoto|[aá]gua|distribuic[aã]o\s+de)\b",
    r"\bestac[aã]o\s+de\s+(tratamento|elevac[aã]o|bombeamento)\b",
    r"\bsubestac[aã]o\s+el[eé]trica\b",
    r"\bestrutura\s+(met[aá]lica|de\s+concreto\s+armado)\b",
    r"\bconcreto\s+armado\b",
    r"\balvenaria\s+estrutural\b",
    r"\bservi[çc]os?\s+de\s+engenharia\b",
    r"\bprojeto\s+(b[aá]sico|executivo|estrutural)\b",
    r"\bart\s+(do\s+)?crea\b|\bcrea[/\s\-]\w{0,3}\b",
    r"\bmemorial\s+descritivo\b",
    r"\bengenharia\s+(civil|el[eé]trica|hidr[aá]ulica|sanit[aá]ria)\b",
    r"\bmanutenc[aã]o\s+(predial|el[eé]trica|hidr[aá]ulica)\b",
    r"\binstalac[aã]o\s+(el[eé]trica|hidr[aá]ulica|de\s+calhas)\b",
]]

def eh_obvio_engenharia(texto):
    if not isinstance(texto, str):
        return False
    return any(rx.search(texto) for rx in PADROES_OBVIO_ENG)

In [ ]:
from collections import defaultdict

# Forma e cor por rótulo — geral=triângulo cinza, engenharia/obras=círculo colorido
SIMBOLO_ROTULO = {'geral': 'triangle-up', 'engenharia': 'circle', 'obras': 'circle'}
COR_ROTULO = {'geral': '#9aa0a6', 'engenharia': '#d62728', 'obras': '#ff7f0e'}

def _arestas_xy(G):
    ex, ey = [], []
    for u, v in G.edges():
        x0, y0 = G.nodes[u]['pos']
        x1, y1 = G.nodes[v]['pos']
        ex += [x0, x1, None]
        ey += [y0, y1, None]
    return ex, ey

def _rotulo_no(G, df, n):
    """Rótulo e texto de hover de um nó."""
    if n < len(df):
        row = df.iloc[n]
        rot = str(row.get('rotulo', '?'))
        return rot, f"[{rot}] {str(row.get('text', ''))[:90]}"
    return '?', str(n)

def show_graph(G, df, color_by='rotulo', titulo='', salvar=None):
    """Grafo limpo: forma por rótulo, cor por rótulo OU cluster, tamanho por grau.

    color_by='rotulo' → legenda geral/engenharia/obras (recomendado p/ achar
    subenquadramento: triângulos cinza no meio de círculos vermelhos = suspeitos).
    color_by='cluster' → cor pelo cluster (vê a estrutura de comunidades).
    salvar → nome (sem extensão) p/ gravar HTML interativo em PASTA_RESULTADOS.
    """
    ex, ey = _arestas_xy(G)
    edge_trace = go.Scatter(x=ex, y=ey, mode='lines', hoverinfo='none',
                            showlegend=False,
                            line=dict(width=0.5, color='rgba(160,160,160,0.20)'))
    graus = dict(G.degree())
    def _sz(n):
        return 7 + (graus.get(n, 1) ** 0.5) * 2.5

    traces = [edge_trace]
    if color_by == 'rotulo':
        grupos = defaultdict(lambda: dict(x=[], y=[], text=[], size=[]))
        for n in G.nodes():
            x, y = G.nodes[n]['pos']
            rot, txt = _rotulo_no(G, df, n)
            g = grupos[rot]
            g['x'].append(x); g['y'].append(y); g['text'].append(txt)
            g['size'].append(_sz(n))
        # ordem: engenharia/obras primeiro (fundo), geral por cima (destaque)
        for rot in ['engenharia', 'obras', 'geral']:
            if rot not in grupos:
                continue
            g = grupos[rot]
            traces.append(go.Scatter(
                x=g['x'], y=g['y'], mode='markers', name=f'{rot} ({len(g["x"])})',
                text=g['text'], hoverinfo='text',
                marker=dict(symbol=SIMBOLO_ROTULO.get(rot, 'circle'),
                            color=COR_ROTULO.get(rot, '#1f77b4'),
                            size=g['size'], line=dict(width=0.5, color='white'),
                            opacity=0.85)))
    else:  # color_by == 'cluster'
        xs, ys, txt, cor, sym, sz = [], [], [], [], [], []
        for n in G.nodes():
            x, y = G.nodes[n]['pos']
            rot, t = _rotulo_no(G, df, n)
            xs.append(x); ys.append(y); txt.append(t)
            cor.append(G.nodes[n].get('cluster', 0))
            sym.append(SIMBOLO_ROTULO.get(rot, 'circle'))
            sz.append(_sz(n))
        traces.append(go.Scatter(
            x=xs, y=ys, mode='markers', text=txt, hoverinfo='text', showlegend=False,
            marker=dict(symbol=sym, color=cor, colorscale='Turbo', size=sz,
                        line=dict(width=0.5, color='white'), opacity=0.85)))

    fig = go.Figure(data=traces, layout=go.Layout(
        title=titulo, showlegend=(color_by == 'rotulo'), hovermode='closest',
        height=620, plot_bgcolor='white',
        legend=dict(itemsizing='constant', title='Rótulo do órgão'),
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig.show()
    if salvar and 'PASTA_RESULTADOS' in globals():
        fig.write_html(f'{PASTA_RESULTADOS}/{salvar}.html')
        print(f'  grafo salvo: {salvar}.html')
    return fig

def show_graph_regularization(G, df, titulo='Propagação de confiança', salvar=None):
    """Mapa de calor do score 'f' (regularização). Forma ainda por rótulo."""
    ex, ey = _arestas_xy(G)
    edge_trace = go.Scatter(x=ex, y=ey, mode='lines', hoverinfo='none',
                            showlegend=False,
                            line=dict(width=0.5, color='rgba(160,160,160,0.20)'))
    xs, ys, txt, cor, sym = [], [], [], [], []
    for n in G.nodes():
        x, y = G.nodes[n]['pos']
        rot, t = _rotulo_no(G, df, n)
        f = float(G.nodes[n].get('f', [0.0])[0])
        xs.append(x); ys.append(y)
        txt.append(f'score={f:.2f}<br>{t}')
        cor.append(f); sym.append(SIMBOLO_ROTULO.get(rot, 'circle'))
    node_trace = go.Scatter(
        x=xs, y=ys, mode='markers', text=txt, hoverinfo='text',
        marker=dict(symbol=sym, color=cor, colorscale='Reds', size=10,
                    line=dict(width=0.5, color='#333'), showscale=True,
                    colorbar=dict(title='confiança')))
    fig = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(
        title=titulo, showlegend=False, hovermode='closest', height=620,
        plot_bgcolor='white', margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig.show()
    if salvar and 'PASTA_RESULTADOS' in globals():
        fig.write_html(f'{PASTA_RESULTADOS}/{salvar}.html')
    return fig

In [ ]:
def simple_regularization(G, labels, max_iter=30):
    """Propagação iterativa: cada nó recebe a média dos vizinhos; nós
    rotulados (seeds) ficam fixos. Resultado em G.nodes[n]['f']."""
    for n in G.nodes():
        G.nodes[n]['f'] = np.array([0.0])
        if n in labels:
            G.nodes[n]['y'] = np.array([1.0])
            G.nodes[n]['f'] = np.array([1.0])
    for it in range(max_iter):
        diff = 0.0
        for node in G.nodes():
            if node in labels:
                continue
            viz = list(G.neighbors(node))
            if not viz:
                continue
            f_new = np.mean([G.nodes[v]['f'] for v in viz], axis=0)
            diff += float(np.linalg.norm(G.nodes[node]['f'] - f_new))
            G.nodes[node]['f'] = f_new
            if 'y' in G.nodes[node]:
                G.nodes[node]['f'] = G.nodes[node]['y']
        print(f'Iteração #{it+1} Q(F)={diff:.4f}')

## 3. Modelo LLM (Google Gemini)

Configura a API e **detecta automaticamente** um modelo válido (os
`gemini-1.5-*` foram descontinuados — daí o erro 404). Define `MODELO_LLM`
usado nas seções 9, 11 e 12.

**Chave:** adicione `GOOGLE_API_KEY` nos Secrets do Colab (ícone 🔑).
Chave gratuita em https://aistudio.google.com/app/apikey

In [ ]:
import os
import google.generativeai as genai
from google.colab import userdata

def llm_local():
    genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))
    print('Gemini API configurada.')

_PREFERIDOS = ['gemini-2.5-flash-lite', 'gemini-2.0-flash-lite',
               'gemini-2.0-flash', 'gemini-2.5-flash', 'gemini-flash-latest',
               'gemini-2.0-flash-001', 'gemini-pro-latest']

def detectar_modelo():
    """Lista modelos disponíveis e devolve o primeiro preferido.
    Preferimos lite/flash-lite porque têm mais quota no free tier (30 RPM
    vs 15 RPM dos models flash padrão) — crucial para evitar 429.
    """
    try:
        disponiveis = [
            m.name.split('/')[-1] for m in genai.list_models()
            if 'generateContent' in getattr(m, 'supported_generation_methods', [])
        ]
    except Exception as e:
        print(f'[llm] não consegui listar modelos ({e}); usando gemini-2.0-flash')
        return 'gemini-2.0-flash'
    for pref in _PREFERIDOS:
        if pref in disponiveis:
            return pref
    flash = [m for m in disponiveis if 'flash' in m]
    return (flash or disponiveis or ['gemini-2.0-flash'])[0]

def llm_task(model, system, prompt, max_tokens=800, temperatura=0.2,
              max_tentativas=3):
    """Chama Gemini com RETRY EXPONENCIAL em rate limit (429).

    Quota free do gemini-flash = 15 RPM. Sem retry, qualquer rajada zera os
    resultados (foi o que aconteceu nos seus vereditos). Backoff: 20s, 40s, 80s.
    """
    for tentativa in range(max_tentativas):
        try:
            gm = genai.GenerativeModel(
                model_name=model,
                generation_config={'temperature': temperatura,
                                   'max_output_tokens': max_tokens})
            resp = gm.generate_content([system, prompt])
            return (resp.text or '').strip()
        except Exception as e:
            msg = str(e)
            eh_rate_limit = '429' in msg or 'quota' in msg.lower() or 'rate' in msg.lower()
            if eh_rate_limit and tentativa < max_tentativas - 1:
                espera = 20 * (2 ** tentativa)  # 20s, 40s, 80s
                print(f'  [rate limit, tentativa {tentativa+1}/{max_tentativas}] '
                      f'aguardando {espera}s...')
                time.sleep(espera)
                continue
            print(f'[llm] erro ({model}): {msg[:140]}')
            return None
    return None

# Sleep entre chamadas para ficar abaixo do RPM do free tier.
# 15 RPM = 1 chamada a cada 4s; usamos 5s p/ margem.
INTERVALO_LLM_SEG = 5

llm_local()
MODELO_LLM = detectar_modelo()
print(f'Modelo LLM selecionado: {MODELO_LLM}')
print(f'Intervalo entre chamadas: {INTERVALO_LLM_SEG}s (free tier ~15 RPM)')

### 3.1 LLM local (Ollama) — recomendado, sem rate limit

O Gemini free (~30 RPM) derruba as células 9, 11.5 e 13 com erro 429. As
duas células abaixo deixam um **LLM local pronto para rodar**: instalam o
Ollama, sobem o servidor, baixam o Llama 3.1 (8B, roda na GPU T4 do Colab) e
redefinem `llm_task` para usá-lo — **sem rate limit**, ~2-5s por chamada.

Rode as duas células para usar o LLM local. Se preferir o Gemini (seção 3),
basta NÃO rodá-las.

In [ ]:
# Instala e sobe o Ollama. Use só se quiser LLM local (sem rate limit).
# Se você tem cota suficiente no Gemini (free ou paga), pule estas duas células.
import subprocess, time, os, requests
from pathlib import Path

def ollama_no_ar(porta=11434):
    try:
        return requests.get(f'http://127.0.0.1:{porta}/api/tags', timeout=2).ok
    except Exception:
        return False

def _existe(cmd):
    return subprocess.run(['which', cmd], capture_output=True,
                           text=True).stdout.strip()

if not ollama_no_ar():
    # Dependências do instalador (lspci é necessário para detectar a GPU;
    # zstd para descompactar o binário).
    if not _existe('lspci') or not _existe('zstd'):
        print('Instalando dependências (lspci, zstd)...')
        subprocess.run('apt-get update -qq && apt-get install -y -qq pciutils zstd',
                       shell=True, check=False)

    if not _existe('ollama'):
        print('Instalando Ollama...')
        r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                           shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print('STDERR:', r.stderr[-500:])
            raise RuntimeError('Falha ao instalar Ollama.')

    ollama_bin = _existe('ollama') or '/usr/local/bin/ollama' or os.path.expanduser(
        '~/.ollama/ollama')
    print(f'Subindo servidor: {ollama_bin} serve')
    subprocess.Popen([ollama_bin, 'serve'],
                     stdout=open('/tmp/ollama.log', 'w'),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if ollama_no_ar():
            break
        time.sleep(1)

if ollama_no_ar():
    subprocess.run(['ollama', 'pull', 'llama3.1'], check=False)
    subprocess.run(['ollama', 'list'], check=False)
    print('Ollama OK no ar:', ollama_no_ar())
else:
    print('⚠ Ollama não subiu. Veja /tmp/ollama.log. Use o Gemini (seção 3).')

In [ ]:
# Redefine llm_task para usar o Ollama local (sem rate limit).
import requests

OLLAMA_MODELO = 'llama3.1'

def llm_task(model, system, prompt, max_tokens=600, temperatura=0.2):
    try:
        r = requests.post('http://127.0.0.1:11434/api/chat', timeout=180,
            json={'model': OLLAMA_MODELO,
                  'messages': [{'role': 'system', 'content': system},
                               {'role': 'user', 'content': prompt}],
                  'stream': False,
                  'options': {'temperature': temperatura,
                              'num_predict': max_tokens}})
        return r.json().get('message', {}).get('content', '').strip() if r.ok else None
    except Exception as e:
        print(f'[llm] {e}')
        return None

MODELO_LLM = OLLAMA_MODELO
INTERVALO_LLM_SEG = 0   # local: sem espera entre chamadas

# Teste rápido
_t = llm_task(MODELO_LLM, 'Responda em uma palavra.', 'Diga OK.')
print('Teste Ollama →', _t if _t else '(sem resposta — verifique a célula anterior)')

## 4. Base textual — contratos PNCP

Lê `contratos.parquet` direto do Drive (sem o pacote `pncp`) e define
`PASTA_RESULTADOS`. Amostra estratificada de até 3000 contratos.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Caminho do consolidado (produzido por uma coleta anterior)
CAMINHO_PARQUET = '/content/drive/MyDrive/PNCP_TCC/dados/coleta/contratos.parquet'
PASTA_RESULTADOS = '/content/drive/MyDrive/PNCP_TCC/resultados_analise_textual'
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
print(f'Resultados salvos em:\n  {PASTA_RESULTADOS}')

df_contratos = pd.read_parquet(
    CAMINHO_PARQUET,
    columns=['numeroControlePNCP', 'objeto', 'rotulo',
             'razaoSocialOrgao', 'municipioNome', 'valor'],
)
df_contratos = df_contratos.rename(columns={'objeto': 'text'})
df_contratos = df_contratos.dropna(subset=['text'])
df_contratos = df_contratos[df_contratos['text'].str.len() > 20].reset_index(drop=True)
print(f'\nBase total: {len(df_contratos):,} contratos.')
print('Distribuição real (sem amostragem):')
print(df_contratos['rotulo'].value_counts().to_string())

# ── Estratégia de amostragem ────────────────────────────────────────────────
# Big data: a base tem >100k contratos. Amostrar perde sinal. Três opções:
#   'tudo'         — usa TODOS os contratos (SBERT pode levar 10+ minutos
#                    em GPU para 100k contratos, mas é o ideal).
#   'estratificado'— N_AMOSTRA por rótulo. PARTE INFERIOR DE GERAL FICA
#                    SUPER-REPRESENTADA → ruim para PU learning.
#   'proporcional' — mantém a proporção real (mais geral, menos eng).
#                    Melhor para refletir a base, OK para classificador
#                    com class_weight='balanced'.
#
# Por que NÃO 'estratificado' (o default antigo)?
#   Quando se sobe N_AMOSTRA mantendo 1000/1000/1000, super-amostramos
#   engenharia (que tem ~6k no PNCP), e o centróide vira tendencioso.
#   Engenharia ∪ obras são tratados como UMA classe positiva no PU
#   (ambos rotulados corretamente como "rito de engenharia exigido").
ESTRATEGIA = 'proporcional'    # 'tudo' | 'estratificado' | 'proporcional'
N_AMOSTRA = 30_000             # ignorado se ESTRATEGIA='tudo'

if ESTRATEGIA == 'tudo' or len(df_contratos) <= N_AMOSTRA:
    df_amostra = df_contratos.copy()
elif ESTRATEGIA == 'estratificado':
    por_classe = N_AMOSTRA // df_contratos['rotulo'].nunique()
    df_amostra = (df_contratos.groupby('rotulo', group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), por_classe),
                                              random_state=42)))
else:  # proporcional
    df_amostra = df_contratos.sample(n=min(N_AMOSTRA, len(df_contratos)),
                                       random_state=42)
df_amostra = df_amostra.reset_index(drop=True)

print(f'\nEstratégia: {ESTRATEGIA} | amostra: {len(df_amostra):,}')
print('Distribuição da amostra:')
print(df_amostra['rotulo'].value_counts().to_string())
df_amostra.head()

## 5. Modelo Espaço-Vetorial + TF-IDF

TF-IDF com tokenizador customizado (stopwords PT + stemmer RSLP), corte por
frequência mínima de documentos (`min_df`).

In [ ]:
VSM = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None, min_df=3)
X = VSM.fit_transform(df_amostra['text'])
print(f'X: {X.shape[0]:,} docs x {X.shape[1]:,} termos (nnz={X.nnz:,})')

df_word_tfidfs = pd.DataFrame({
    'word': VSM.get_feature_names_out(),
    'tfidf_sum': X.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_word_tfidfs.head(50)

## 6. N-gramas (bigramas e trigramas)

Termos compostos relevantes (ex.: *projeto básico*, *obra civil*,
*manutenção predial*) — vocabulário que denuncia engenharia escondida.

In [ ]:
VSM_bi = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                          min_df=3, ngram_range=(2, 2))
X_bi = VSM_bi.fit_transform(df_amostra['text'])
df_bi = pd.DataFrame({
    'word': VSM_bi.get_feature_names_out(),
    'tfidf_sum': X_bi.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_bi.head(30)

In [ ]:
VSM_tri = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                           min_df=3, ngram_range=(3, 3))
X_tri = VSM_tri.fit_transform(df_amostra['text'])
df_tri = pd.DataFrame({
    'word': VSM_tri.get_feature_names_out(),
    'tfidf_sum': X_tri.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_tri.head(30)

## 7. Rede k-NN sobre TF-IDF + Label Propagation

Conecta contratos similares (k-NN sobre TF-IDF 1+2-gramas) e detecta
comunidades por label propagation.

In [ ]:
VSM = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                       min_df=3, ngram_range=(1, 2))
X = VSM.fit_transform(df_amostra['text'])
A = kneighbors_graph(X, n_neighbors=5, metric='cosine')
G = nx.Graph(A)

cluster_id = 0
for clusters in community.label_propagation_communities(G):
    for doc_id in clusters:
        G.nodes[doc_id]['cluster'] = cluster_id
    cluster_id += 1

df_amostra['cluster'] = [G.nodes[i].get('cluster', -1) for i in df_amostra.index]
df_amostra.head()

In [ ]:
# Estatísticas por cluster
df_stats = (df_amostra[['text', 'cluster']].groupby('cluster').count()
            .sort_values(by='text', ascending=False)
            .rename(columns={'text': 'num_docs'}))
df_stats.head(20)

### 7.1 Descritores de cluster (termos com maior TF-IDF)

In [ ]:
QTD_TOPICOS = 15
linhas = []
for c in df_stats.head(QTD_TOPICOS).index:
    n_docs, descs = get_cluster_descriptors(VSM, df_amostra, c)
    linhas.append([c, n_docs, descs])
df_descriptors = pd.DataFrame(linhas, columns=['cluster', 'num_docs', 'descriptors'])
df_descriptors

### 7.2 Subgrafo dos maiores clusters + visualização

Triângulos cinza (`geral`) próximos de círculos vermelhos (`engenharia`) já
sugerem rotulação suspeita.

In [ ]:
selected = df_descriptors.cluster.tolist()
G2 = G.copy()
for n in list(G2.nodes()):
    if G2.nodes[n].get('cluster') not in selected:
        G2.remove_node(n)

pos = nx.spring_layout(G2, seed=42, k=0.5)  # k maior = nós mais espalhados
for n in G2.nodes():
    G2.nodes[n]['pos'] = pos[n]

print(f'Subgrafo TF-IDF: {G2.number_of_nodes()} nós, {G2.number_of_edges()} arestas')
# Vista por CLUSTER (cor = comunidade). Forma ainda indica o rótulo
# (triângulo = geral). Difere do grafo SBERT (colorido por rótulo) — aqui
# o foco é ver a ESTRUTURA de comunidades do TF-IDF.
show_graph(G2, df_amostra, color_by='cluster',
           titulo='Rede k-NN (TF-IDF) — cor por cluster, forma por rótulo',
           salvar='grafo_tfidf_cluster')

## 8. Embeddings contextuais (Sentence-BERT multilíngue)

DistilBERT multilíngue ajustado para similaridade semântica. Captura sentido
além das palavras exatas — base para a detecção da seção 11.

In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('distiluse-base-multilingual-cased-v1')

# IMPORTANTE: aplica limpar_boilerplate antes do encode.
# Sem isso, o SBERT vê "CONTRATAÇÃO DE EMPRESA ESPECIALIZADA PARA PRESTAÇÃO
# DE SERVIÇOS DE ..." em quase TODO contrato — esse prefixo dominado e
# idêntico arrasta a similaridade para cima artificialmente e o método
# de centróide/kNN produz falsos positivos (foi a causa dos top suspeitos
# do run anterior virem cheios de geral legítimo: locução, alimentação,
# perícia contábil etc.).
textos_para_emb = df_amostra['text'].astype(str).map(limpar_boilerplate).tolist()
print('Exemplo de limpeza (3 primeiros):')
for orig, limpo in zip(df_amostra['text'].head(3), textos_para_emb[:3]):
    print(f'  antes:  {orig[:90]}')
    print(f'  depois: {limpo[:90]}\n')

X_emb = sbert.encode(textos_para_emb, show_progress_bar=True,
                       batch_size=64, convert_to_numpy=True,
                       normalize_embeddings=True)
X_emb.shape

In [ ]:
A_emb = kneighbors_graph(X_emb, n_neighbors=5, metric='cosine')
G_emb = nx.Graph(A_emb)

cid = 0
for clusters in community.label_propagation_communities(G_emb):
    for doc_id in clusters:
        G_emb.nodes[doc_id]['cluster'] = cid
    cid += 1
df_amostra['cluster_sbert'] = [G_emb.nodes[i].get('cluster', -1)
                                  for i in df_amostra.index]

df_stats_emb = (df_amostra[['text', 'cluster_sbert']]
                .groupby('cluster_sbert').count()
                .sort_values('text', ascending=False)
                .rename(columns={'text': 'num_docs'}))
df_stats_emb.head(20)

### 8.1 Subgrafo semântico (top-20 clusters) + visualização

In [ ]:
# Reduz para os 20 maiores clusters semânticos (50 polui demais)
selected_emb = list(df_stats_emb.head(20).index)
G_emb2 = G_emb.copy()
for n in list(G_emb2.nodes()):
    if G_emb2.nodes[n].get('cluster') not in selected_emb:
        G_emb2.remove_node(n)

pos = nx.spring_layout(G_emb2, seed=42, k=0.6)
for n in G_emb2.nodes():
    G_emb2.nodes[n]['pos'] = pos[n]

print(f'Subgrafo semântico: {G_emb2.number_of_nodes()} nós, '
      f'{G_emb2.number_of_edges()} arestas (top-20 clusters)')
show_graph(G_emb2, df_amostra, color_by='rotulo',
           titulo='Rede semântica (SBERT) — forma/cor por rótulo do órgão',
           salvar='grafo_sbert_rotulo')

## 9. Indicadores por cluster (LLM)

Para cada cluster grande, o LLM lê uma amostra dos objetos e devolve um
indicador estruturado (nome, categoria, indício de subenquadramento).

In [ ]:
SYSTEM_INDICADOR = '''
Você é analista de contratações públicas brasileiras (Lei 14.133/2021).

Recebe uma amostra de objetos de contratos agrupados por similaridade
semântica. Identifique UM indicador-chave que sintetize o padrão.

ATENÇÃO: considere apenas ENGENHARIA (CREA/ART) e OBRAS. Arquitetura
(CAU/RRT) NÃO entra na análise.

Sobre o campo "indicio_subenquadramento":
- "alto"   → o cluster é claramente de engenharia/obras E a amostra contém
             contratos que NÃO deveriam estar rotulados como "geral".
             Também use "alto" para clusters que SÃO de engenharia/obras
             puros — eles representam o vocabulário de referência usado
             para detectar subenquadramento na base "geral".
- "medio"  → cluster majoritariamente técnico mas com fronteira ambígua
             (ex.: manutenção de equipamentos que pode ou não exigir CREA).
- "baixo"  → cluster predominantemente de serviços comuns (limpeza,
             alimentação, vigilância) com poucos casos limítrofes.
- "nenhum" → cluster sem nenhuma relação com engenharia/obras.

Responda APENAS com este JSON plano (uma linha, sem aninhamento):
{"nome": "nome curto e descritivo", "categoria": "obra civil|manutencao predial|servico de engenharia|vigilancia/limpeza|fornecimento|transporte|outro", "indicio_subenquadramento": "alto|medio|baixo|nenhum", "descricao": "1-2 frases sobre o padrao e por que importa para auditoria"}'''

In [ ]:
TOP_N_CLUSTERS = 12
N_OBJETOS_POR_CLUSTER = 8

def _json_do_texto(txt):
    """Extrai JSON, tolerando ```json fences e texto ao redor."""
    if not txt:
        return None
    t = txt.replace('```json', '').replace('```', '')
    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

indicadores = []
for j, cid in enumerate(df_stats_emb.head(TOP_N_CLUSTERS).index):
    if j > 0:
        time.sleep(INTERVALO_LLM_SEG)   # respeita rate limit
    sub = df_amostra[df_amostra.cluster_sbert == cid].head(N_OBJETOS_POR_CLUSTER)
    objetos = '\n'.join(f'- {str(o)[:300]}' for o in sub['text'].tolist())
    prompt = (f'Cluster #{cid} ({len(sub)} objetos amostrados).\n\n'
              f'Objetos:\n{objetos}\n\nGere o indicador no JSON plano.')
    resp = llm_task(MODELO_LLM, SYSTEM_INDICADOR, prompt, max_tokens=400)
    ind = _json_do_texto(resp) or {}
    indicadores.append({
        'cluster': cid,
        'n_docs': int((df_amostra.cluster_sbert == cid).sum()),
        'nome': ind.get('nome', '?'),
        'categoria': ind.get('categoria', '?'),
        'indicio_subenq': ind.get('indicio_subenquadramento', '?'),
        'descricao': str(ind.get('descricao', ''))[:200],
    })
    print(f'  {j+1}/{TOP_N_CLUSTERS} cluster #{cid}: {indicadores[-1]["nome"][:55]} '
          f'[{indicadores[-1]["indicio_subenq"]}]')

df_indicadores = pd.DataFrame(indicadores)
print(f'\n{len(df_indicadores)} indicadores gerados pelo LLM.')
df_indicadores.sort_values('indicio_subenq',
                            key=lambda s: s.map({'alto': 0, 'medio': 1,
                                                  'baixo': 2, 'nenhum': 3}).fillna(4))

## 10. Classificação semissupervisionada (regularização em grafo)

Método complementar: a partir de poucos *seeds* (contratos `geral` com termos
óbvios de engenharia), propaga a confiança pelos vizinhos do grafo semântico.

In [ ]:
TERMOS_OBVIOS = ('art ', 'crea', 'projeto básico', 'projeto basico',
                  'obra de engenharia', 'reforma predial', 'pavimentação',
                  'pavimentacao', 'memorial descritivo', 'abnt nbr')

labels = {}
for n in G_emb2.nodes():
    objeto = str(df_amostra.loc[n, 'text']).lower()
    rotulo = str(df_amostra.loc[n, 'rotulo'])
    if rotulo == 'geral' and any(t in objeto for t in TERMOS_OBVIOS):
        labels[n] = 1

print(f'Seeds encontradas (geral + termo óbvio): {len(labels)}')
for n in list(labels)[:5]:
    print(f'  #{n}: {str(df_amostra.loc[n, "text"])[:120]}')

In [ ]:
if labels:
    simple_regularization(G_emb2, labels, max_iter=30)
    show_graph_regularization(G_emb2, df_amostra,
                               titulo='Regularização — confiança propagada a partir dos seeds',
                               salvar='grafo_regularizacao')
else:
    print('Sem seeds — refine TERMOS_OBVIOS ou aumente a amostra.')

### 10.1 Filtrando candidatos por limiar

In [ ]:
LIMIAR = 0.5
candidatos = []
for n in G_emb2.nodes():
    f = G_emb2.nodes[n].get('f', np.array([0.0]))
    if float(f[0]) > LIMIAR and n not in labels:
        candidatos.append(n)

df_candidatos = df_amostra.loc[candidatos, ['numeroControlePNCP', 'rotulo',
                                              'text', 'municipioNome', 'valor']]
print(f'Candidatos com score > {LIMIAR}: {len(df_candidatos)}')
df_candidatos.head(20)

## 11. Detecção de subenquadramento usando os rótulos limpos — FOCO PRINCIPAL

**Hipótese metodológica:** os rótulos `engenharia` e `obras` têm alta precisão
(quando o órgão marcou assim, é porque é) — mas o rótulo `geral` é ruidoso:
parte é serviço comum de verdade, parte é engenharia mal classificada.

**Objetivo:** ranquear os contratos `geral` pela probabilidade de serem na
verdade engenharia/obras, usando 3 sinais complementares:

1. **Similaridade ao centróide** das embeddings de `engenharia + obras`
2. **Voto k-NN** — qual é a maioria dos vizinhos no grafo semântico?
3. **Pureza do cluster** — clusters dominados por engenharia/obras

> Nota: aqui identificamos apenas a **rotulação incorreta**. A verificação do
> rito de engenharia (ART/RRT, projeto básico, etc.) fica para etapa futura
> sobre os PDFs.

### 11.1 Centróide das classes "limpas" (engenharia + obras)

Calculamos o centróide das embeddings de `engenharia ∪ obras` — o "vetor médio"
do que é engenharia — e medimos a similaridade de cada `geral` a ele. Os mais
próximos são candidatos a estarem mal rotulados.

**Setup PU (Positive-Unlabeled):** `engenharia ∪ obras` = positivos limpos;
`geral` = conjunto misto (negativos reais + falsos negativos a encontrar).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from sklearn.linear_model import LogisticRegression

mask_pos = df_amostra['rotulo'].isin(['engenharia', 'obras'])
mask_geral = df_amostra['rotulo'] == 'geral'
print(f'Positivos limpos (engenharia + obras): {mask_pos.sum()}')
print(f'"geral" a investigar:                  {mask_geral.sum()}')

# ── Sinal contínuo 1: similaridade ao centróide (referência simples) ────────
centroide_pos = X_emb[mask_pos.values].mean(axis=0)
centroide_pos = centroide_pos / (np.linalg.norm(centroide_pos) + 1e-12)
sims = cosine_similarity(X_emb[mask_geral.values],
                          centroide_pos.reshape(1, -1)).ravel()

# ── Sinal contínuo 2: CLASSIFICADOR supervisionado (fronteira nítida) ───────
# PU baseline: eng+obras = 1, geral = 0 (geral é "negativo ruidoso"). O
# classificador aprende a FRONTEIRA entre as classes e dá P(eng/obras) para
# cada geral. É muito mais discriminativo que a distância ao centróide — o
# centróide confunde "serviço genérico" (alimentação, limpeza, perícia) com
# engenharia porque todos ficam a meia distância da média. A fronteira do
# classificador separa melhor o que é tecnicamente engenharia.
y = mask_pos.astype(int).values
clf_pu = LogisticRegression(max_iter=3000, class_weight='balanced', C=1.0)
clf_pu.fit(X_emb, y)
prob_geral = clf_pu.predict_proba(X_emb[mask_geral.values])[:, 1]

df_geral = df_amostra[mask_geral].copy().reset_index(drop=True)
df_geral['sim_centroide_eng'] = sims
df_geral['prob_eng_clf'] = prob_geral

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(sims, bins=60, color='#1f77b4')
axes[0].set_title('Sinal 1 — similaridade ao centróide eng/obras')
axes[0].set_xlabel('cosseno'); axes[0].set_ylabel('nº de "geral"')
axes[1].hist(prob_geral, bins=60, color='#2ca02c')
axes[1].axvline(0.5, color='#d62728', ls='--', lw=0.8, label='0.5')
axes[1].set_title('Sinal 2 — P(eng/obras) do classificador')
axes[1].set_xlabel('probabilidade'); axes[1].legend()
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/sinais_continuos.png', dpi=120,
            bbox_inches='tight')
plt.show()

print('\nTop 15 "geral" por P(eng/obras) do classificador '
      '(sinal mais discriminativo):')
df_geral.sort_values('prob_eng_clf', ascending=False).head(15)[
    ['numeroControlePNCP', 'text', 'prob_eng_clf', 'sim_centroide_eng']]

### 11.2 Voto k-NN — vizinhos com rótulo confiável

Para cada `geral` no grafo k-NN do SBERT, vemos o rótulo dos vizinhos. Maioria
engenharia/obras → provavelmente mal rotulado.

In [ ]:
votos = []
for n in G_emb.nodes():
    if n >= len(df_amostra):
        continue
    rotulo_n = df_amostra.iloc[n]['rotulo']
    if rotulo_n != 'geral':
        continue
    vizinhos = list(G_emb.neighbors(n))
    vizinhos_validos = [v for v in vizinhos if v < len(df_amostra)]
    if not vizinhos_validos:
        continue
    rot_viz = df_amostra.iloc[vizinhos_validos]['rotulo'].value_counts()
    n_eng = int(rot_viz.get('engenharia', 0) + rot_viz.get('obras', 0))
    n_geral_viz = int(rot_viz.get('geral', 0))
    n_total = n_eng + n_geral_viz
    if n_total == 0:
        continue
    votos.append({
        'numeroControlePNCP': df_amostra.iloc[n]['numeroControlePNCP'],
        'objeto': str(df_amostra.iloc[n]['text'])[:120],
        'n_vizinhos_eng_obras': n_eng,
        'n_vizinhos_geral': n_geral_viz,
        'pct_vizinhos_eng_obras': n_eng / n_total,
        'n_vizinhos': n_total,
    })
df_votos = (pd.DataFrame(votos)
            .sort_values('pct_vizinhos_eng_obras', ascending=False)
            .reset_index(drop=True))
print(f'{len(df_votos)} contratos "geral" com vizinhos analisados')
print(f'Com ≥50% dos vizinhos eng/obras: '
      f'{(df_votos["pct_vizinhos_eng_obras"] >= 0.5).sum()}')
df_votos.head(15)

### 11.3 Pureza de cluster — clusters dominados por engenharia/obras

Clusters do SBERT com predominância de engenharia/obras + alguns `geral` no
meio → esses `geral` são candidatos fortes.

In [ ]:
pureza = (df_amostra.groupby('cluster_sbert')['rotulo']
          .value_counts(normalize=False).unstack(fill_value=0))
for col in ('engenharia', 'obras', 'geral'):
    if col not in pureza.columns:
        pureza[col] = 0
pureza['total'] = pureza[['engenharia', 'obras', 'geral']].sum(axis=1)
pureza['pct_eng_obras'] = (
    (pureza['engenharia'] + pureza['obras']) / pureza['total'].clip(lower=1)
)

# Cluster suspeito: ≥60% engenharia/obras + ≥1 "geral" no meio + ≥5 contratos
clusters_suspeitos = pureza[
    (pureza['pct_eng_obras'] >= 0.6)
    & (pureza['geral'] >= 1)
    & (pureza['total'] >= 5)
].sort_values('pct_eng_obras', ascending=False)

print(f'Clusters suspeitos (eng/obras dominam, mas têm "geral" dentro): '
      f'{len(clusters_suspeitos)}')
clusters_suspeitos[['engenharia', 'obras', 'geral', 'total',
                     'pct_eng_obras']].head(20)

### 11.4 Consolidação — ranking por nº de sinais positivos

Soma dos três sinais (0–3). Quanto mais sinais para um `geral`, maior a chance
de ser engenharia/obras mal rotulado. Gráficos: distribuição de sinais e
dispersão dos dois sinais contínuos.

In [ ]:
import matplotlib.pyplot as plt

# Limiares dos sinais. O classificador é o sinal forte (fronteira nítida).
LIMIAR_CLF = 0.60          # P(eng/obras) >= 0.60
LIMIAR_VOTO = 0.5          # >=50% dos vizinhos kNN são eng/obras

df_ranking = df_geral.merge(
    df_votos[['numeroControlePNCP', 'pct_vizinhos_eng_obras', 'n_vizinhos']],
    on='numeroControlePNCP', how='left'
)
df_ranking['no_cluster_suspeito'] = df_ranking['cluster_sbert'].isin(
    clusters_suspeitos.index)

# 4 sinais binários complementares
df_ranking['sinal_clf'] = (df_ranking['prob_eng_clf'] >= LIMIAR_CLF).astype(int)
df_ranking['sinal_voto_knn'] = (
    df_ranking['pct_vizinhos_eng_obras'].fillna(0) >= LIMIAR_VOTO).astype(int)
df_ranking['sinal_cluster'] = df_ranking['no_cluster_suspeito'].astype(int)
# Regex de alta precisão (independente de LLM/embeddings)
df_ranking['sinal_obvio_eng'] = (
    df_ranking['text'].astype(str).map(eh_obvio_engenharia)).astype(int)

df_ranking['n_sinais'] = (df_ranking['sinal_clf']
                            + df_ranking['sinal_voto_knn']
                            + df_ranking['sinal_cluster']
                            + df_ranking['sinal_obvio_eng'])

# Ordena por nº de sinais e, como desempate, pela probabilidade do classificador
df_ranking = df_ranking.sort_values(
    ['n_sinais', 'prob_eng_clf'], ascending=False).reset_index(drop=True)

# Dedup: pregões/lotes diferentes do mesmo órgão geram NCPs distintos para
# objetos quase idênticos. Chave = CNPJ + primeiros 80 chars do objeto.
def _chave_dedup(row):
    ncp = str(row.get('numeroControlePNCP', ''))
    cnpj = ncp.split('-')[0] if '-' in ncp else ncp[:14]
    obj = str(row['text']).lower().strip()[:80]
    return f'{cnpj}|{obj}'

n_antes = len(df_ranking)
df_ranking['_dedup'] = df_ranking.apply(_chave_dedup, axis=1)
df_ranking = (df_ranking.drop_duplicates('_dedup', keep='first')
              .drop(columns='_dedup').reset_index(drop=True))
print(f'Dedup: {n_antes} → {len(df_ranking)} '
      f'({n_antes - len(df_ranking)} duplicatas removidas)')

# ── Visualização ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
cont = df_ranking['n_sinais'].value_counts().sort_index()
cores_barra = ['#9aa0a6', '#ffd24d', '#ff8c42', '#d62728', '#7e0202']
ax1.bar(cont.index.astype(str), cont.values,
        color=[cores_barra[min(i, 4)] for i in cont.index])
for i, v in zip(cont.index, cont.values):
    ax1.text(str(i), v, str(v), ha='center', va='bottom', fontsize=10)
ax1.set_title('Contratos "geral" por nº de sinais (0–4)')
ax1.set_xlabel('nº de sinais'); ax1.set_ylabel('nº de contratos')

sc = ax2.scatter(df_ranking['prob_eng_clf'],
                  df_ranking['pct_vizinhos_eng_obras'].fillna(0),
                  c=df_ranking['n_sinais'], cmap='YlOrRd', s=18, alpha=0.7)
ax2.axvline(LIMIAR_CLF, color='gray', ls='--', lw=0.8)
ax2.axhline(LIMIAR_VOTO, color='gray', ls='--', lw=0.8)
ax2.set_xlabel('P(eng/obras) — classificador')
ax2.set_ylabel('% vizinhos eng/obras (kNN)')
ax2.set_title('Sinais contínuos (cor = nº de sinais)')
plt.colorbar(sc, ax=ax2, label='n_sinais')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/sinais_subenquadramento.png', dpi=120,
            bbox_inches='tight')
plt.show()

print('\nDistribuição de "geral" por nº de sinais:')
print(cont.to_string())
print(f'\nCandidatos com >=2 sinais: {(df_ranking["n_sinais"] >= 2).sum()}')
print(f'Candidatos com >=3 sinais (alta confiança): '
      f'{(df_ranking["n_sinais"] >= 3).sum()}')
print(f'Candidatos com 4 sinais (todos batem): '
      f'{(df_ranking["n_sinais"] == 4).sum()}')

df_ranking.head(30)[['numeroControlePNCP', 'text', 'prob_eng_clf',
                       'pct_vizinhos_eng_obras', 'sinal_obvio_eng',
                       'cluster_sbert', 'n_sinais']]

### 11.5 Validação final dos top suspeitos pelo LLM

O LLM dá o veredicto sobre a CLASSE CORRETA do objeto (engenharia/obras/geral),
sem julgar rito legal. Confirma ou descarta cada top suspeito.

In [ ]:
import matplotlib.pyplot as plt

def _json_do_texto(txt):
    if not txt:
        return None
    t = txt.replace('```json', '').replace('```', '')
    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

SYSTEM_ROTULACAO = """Você é analista de contratações públicas brasileiras
(Lei 14.133/2021). Classifique APENAS o TIPO do contrato pelo objeto. NÃO
julgue violação legal nem rito formal.

Considere SOMENTE engenharia (CREA/ART) e obras. Arquitetura (CAU/RRT) NÃO
entra: serviço puramente arquitetônico é "geral" neste estudo.

REGRA DE OURO — só é engenharia/obras quando há EXECUÇÃO de serviço técnico
ou intervenção física. NÃO são engenharia/obras (são "geral"), mesmo com tema
de engenharia:
- LOCAÇÃO/ALUGUEL de equipamento (retroescavadeira, plataforma, andaime)
- AQUISIÇÃO/FORNECIMENTO de bens ou materiais (toldos, bandeiras, granito,
  gás, energia elétrica, água)
- PARTICIPAÇÃO em evento/curso/workshop/palestra (mesmo de engenharia)
- Serviços administrativos, jurídicos, contábeis, culturais, de eventos
- Plano/estudo meramente administrativo (ex.: plano de saneamento como peça
  de planejamento, sem execução)

Classes:
- "engenharia": EXECUÇÃO de serviço técnico que exige profissional do CREA —
  reparo/manutenção predial estrutural, elétrica predial, hidráulica predial,
  instalação técnica predial (combate a incêndio, ar-condicionado predial),
  projeto/laudo/levantamento de engenharia, drenagem, sondagem.
- "obras": intervenção física construtiva — construção, reforma estrutural,
  ampliação de edificação, pavimentação asfáltica, demolição, restauro de
  monumento/edificação.
- "geral": tudo o mais (ver regra de ouro).

Responda APENAS no JSON:
{"classe_correta": "engenharia"|"obras"|"geral",
 "confianca": 0.0-1.0,
 "motivo": "justificativa de até 40 palavras explicando por quê"}"""

N_VEREDITO = 30
top = df_ranking.head(N_VEREDITO).reset_index(drop=True)

veredictos = []
for i, row in top.iterrows():
    if i > 0:
        time.sleep(INTERVALO_LLM_SEG)
    obj = str(row['text'])[:600]
    resp_txt = llm_task(MODELO_LLM, SYSTEM_ROTULACAO, f"Objeto: {obj}")
    parsed = _json_do_texto(resp_txt) or {}
    veredictos.append({
        'numeroControlePNCP': row['numeroControlePNCP'],
        'objeto': obj[:200],
        'rotulo_original': row.get('rotulo'),
        'n_sinais': row['n_sinais'],
        'prob_eng_clf': round(float(row.get('prob_eng_clf', 0)), 3),
        'sinal_obvio_eng': row.get('sinal_obvio_eng', 0),
        'llm_classe': parsed.get('classe_correta', '?'),
        'llm_confianca': float(parsed.get('confianca', 0) or 0),
        'llm_motivo': str(parsed.get('motivo', ''))[:400],
    })
    if (i + 1) % 5 == 0:
        print(f'  {i+1}/{N_VEREDITO} processados...')

df_veredictos = pd.DataFrame(veredictos)
# Confirmado = LLM diz eng/obras com confiança >= 0.6.
# NÃO usamos mais a regex óbvia como confirmação automática (ela é só um
# sinal de pré-filtragem; o LLM com regra de ouro é o juiz final).
df_veredictos['confirmado_subenq'] = (
    df_veredictos['llm_classe'].isin(['engenharia', 'obras']) &
    (df_veredictos['llm_confianca'] >= 0.6)
)
n_conf = int(df_veredictos['confirmado_subenq'].sum())
taxa = n_conf / max(len(df_veredictos), 1)

fig, ax = plt.subplots(figsize=(7, 4))
cont_llm = df_veredictos['llm_classe'].value_counts()
cores = {'engenharia': '#d62728', 'obras': '#ff7f0e', 'geral': '#9aa0a6',
         '?': '#cccccc'}
ax.bar(cont_llm.index, cont_llm.values,
       color=[cores.get(c, '#1f77b4') for c in cont_llm.index])
for c, v in zip(cont_llm.index, cont_llm.values):
    ax.text(c, v, str(v), ha='center', va='bottom')
ax.set_title(f'Veredito LLM sobre top {N_VEREDITO} suspeitos\n'
             f'{n_conf} confirmados como engenharia/obras ({taxa:.0%})')
ax.set_ylabel('nº de contratos')
fig.savefig(f'{PASTA_RESULTADOS}/veredito_llm.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nConfirmados subenquadramento: {n_conf}/{N_VEREDITO} ({taxa:.0%})')
# Mostra com a justificativa COMPLETA (sem cortar na exibição)
pd.set_option('display.max_colwidth', 200)
df_veredictos[['numeroControlePNCP', 'objeto', 'llm_classe', 'llm_confianca',
                'llm_motivo', 'confirmado_subenq']]

## 12. Análise de rito de engenharia — quem é violação vs. erro de cadastro

Pega os top suspeitos do ranking (seção 11), baixa os PDFs anexados no PNCP
(TR, Projeto Básico, ETP, Edital, Contrato) e detecta 8 marcadores legais
(ART, CREA, Engenheiro Responsável, ATC, Projeto Básico, Obra/Serv. Eng.,
Norma ABNT NBR, Lei 14.133 art. 6º).

**Veredito jurídico final** (separando rotulação errada de violação real):

- `subenquadramento_real` — objeto óbvio engenharia + rito NÃO seguido →
  provável **violação da Lei 14.133/2021**
- `rotulacao_incorreta_processo_ok` — objeto óbvio engenharia + rito seguido
  → erro de cadastro, mas o processo correu como engenharia
- `sem_documento` — contrato sem PDF anexado (não conclusivo)

> Reusa o pacote `pncp` do repositório clonado — mesma infraestrutura do
> pipeline completo, mas aplicada aos suspeitos encontrados aqui.

### 12.1 Setup — carrega o pacote pncp

In [ ]:
import sys
from pathlib import Path

# Força o repo clonado à frente de qualquer pncp instalado (evita versão velha).
REPO_DIR = '/content/pncp-engineering-analytics'
if not Path(REPO_DIR).exists():
    import subprocess
    subprocess.run(['git', 'clone',
                    'https://github.com/caiofvserra/pncp-engineering-analytics',
                    REPO_DIR], check=False)
if REPO_DIR in sys.path:
    sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)
# Limpa módulos pncp já carregados (caso uma versão instalada tenha entrado)
for _m in [m for m in list(sys.modules) if m == 'pncp' or m.startswith('pncp.')]:
    del sys.modules[_m]

try:
    from pncp import pdfs as pncp_pdfs
    from pncp._marcadores import (
        MAPA_TIPO_DOCUMENTO, TIPOS_RELEVANTES_ENGENHARIA,
        detectar_marcadores, normalizar_pdf_text, COLS_PRESENCA,
    )
    PNCP_OK = True
    print(f'pacote pncp carregado de {pncp_pdfs.__file__}')
except Exception as e:
    PNCP_OK = False
    print(f'pacote pncp indisponível ({e})')
    print(f'  rode: !git clone https://github.com/caiofvserra/'
          f'pncp-engineering-analytics {REPO_DIR}')

### 12.2 Baixa PDFs e detecta marcadores legais

Usa a API de integração do PNCP. Cada contrato leva 5–30s. Cacheia PDFs em
`resultados_analise_textual/rito/cache_pdfs/`.

In [ ]:
# ── Seleção dos suspeitos para análise de rito ──────────────────────────────
# Big data: seleciona por LIMIAR de probabilidade do classificador.
LIMIAR_PROB_RITO = 0.70
MAX_RITO = 500

base_rito = (df_ranking[df_ranking['prob_eng_clf'] >= LIMIAR_PROB_RITO]
             .sort_values('prob_eng_clf', ascending=False)
             .head(MAX_RITO)
             .reset_index(drop=True))

PASTA_RITO = Path(PASTA_RESULTADOS) / 'rito'
PASTA_RITO.mkdir(parents=True, exist_ok=True)
CACHE_PDFS = PASTA_RITO / 'cache_pdfs'
CACHE_PDFS.mkdir(exist_ok=True)

print(f'Analisando rito de {len(base_rito)} suspeitos com '
      f'P(eng/obras) >= {LIMIAR_PROB_RITO} (de {len(df_ranking)} no ranking).')
print(f'PDFs cacheados em: {CACHE_PDFS}\n')

CHARS_MIN_LEGIVEL = 300

def _path_cache(ncp, seq_doc):
    return CACHE_PDFS / f'{ncp.replace("/", "_")}_{seq_doc}.pdf'

def _seq_doc(d):
    return (d.get('sequencialDocumento') or d.get('sequencial')
            or d.get('sequencialArquivo') or d.get('id'))

# ── DIAGNÓSTICO FORTE no início ─────────────────────────────────────────────
# Cobre as 5 primeiras chamadas: status HTTP, URL, e — se houve docs — os
# campos retornados pela API. Sem isso é cego o motivo de "baixados=0".
def _diag_listagem(ncp, info, recurso, docs, status):
    """Diagnóstico de uma chamada de listagem."""
    url = (f'https://pncp.gov.br/api/pncp/v1/orgaos/{info["cnpj"]}/{recurso}/'
           f'{info["ano"]}/{info["sequencial"]}/arquivos')
    print(f'  • NCP={ncp} recurso={recurso} status={status} '
          f'docs={len(docs)} url={url}')
    if docs:
        d0 = docs[0]
        print(f'    campos do 1º doc: {list(d0.keys())}')
        amostra = {k: d0.get(k) for k in list(d0)[:6]}
        print(f'    amostra: {amostra}')

# ── Fluxo: para cada NCP, tenta a listagem direta. Se vier vazia E for
# contrato (tipo 2), tenta também como compra (tipo 1) — útil porque o
# TR/PB anexado costuma estar na compra-mãe, não no contrato. ──────────────
def listar_com_fallback(info, ncp):
    """Tenta primeiro o recurso esperado, depois o outro."""
    recurso_principal = 'compras' if info['tipo'] == 1 else 'contratos'
    docs, status = pncp_pdfs._listar_documentos(
        info['cnpj'], info['ano'], info['sequencial'], recurso_principal)
    if status == 'ok' and not docs:
        # Tenta o outro recurso (compra ↔ contrato no mesmo cnpj/ano/seq)
        outro = 'contratos' if recurso_principal == 'compras' else 'compras'
        docs2, status2 = pncp_pdfs._listar_documentos(
            info['cnpj'], info['ano'], info['sequencial'], outro)
        if status2 == 'ok' and docs2:
            return docs2, status2, outro
    return docs, status, recurso_principal

resultados_rito = []
n_sem_doc = n_baixados = n_cache = n_erro = n_sem_pdf_legivel = 0
n_diag = 0

for i, row in base_rito.iterrows():
    ncp = str(row['numeroControlePNCP'])
    info = pncp_pdfs._decompor_ncp(ncp)
    if not info or info['tipo'] not in (1, 2):
        continue
    docs, status, recurso = listar_com_fallback(info, ncp)

    if n_diag < 5:
        _diag_listagem(ncp, info, recurso, docs, status)
        n_diag += 1

    reg = {
        'numeroControlePNCP': ncp, 'objeto': str(row['text'])[:250],
        'recurso_usado': recurso,
        'n_sinais': int(row.get('n_sinais', 0)),
        'prob_eng_clf': round(float(row.get('prob_eng_clf', 0)), 3),
        'n_pdfs_processados': 0, 'tipos_pdf': '', 'chars_total': 0,
        'mk_score_engenharia': 0, 'seguiu_rito': False,
    }
    for c in COLS_PRESENCA:
        reg[c] = False

    if status != 'ok':
        n_erro += 1
        reg['classificacao_rito'] = 'indeterminado_erro_api'
        resultados_rito.append(reg)
        continue
    if not docs:
        n_sem_doc += 1
        reg['classificacao_rito'] = 'indeterminado_sem_documento'
        resultados_rito.append(reg)
        continue

    # Permissivo: prioriza tipos relevantes mas NÃO exclui nenhum doc.
    ordem = {t: k for k, t in enumerate(TIPOS_RELEVANTES_ENGENHARIA)}
    docs_ord = sorted(docs, key=lambda d: ordem.get(d.get('tipoDocumentoId'), 999))

    marc_uni = {c: False for c in COLS_PRESENCA}
    tipos, chars = [], 0
    for d in docs_ord[:4]:
        seq_doc = _seq_doc(d)
        if not seq_doc:
            continue
        cache = _path_cache(ncp, seq_doc)
        if cache.exists() and cache.stat().st_size > 0:
            n_cache += 1
        else:
            ok = pncp_pdfs._baixar(info['cnpj'], info['ano'],
                                    info['sequencial'], seq_doc, cache, recurso)
            if not ok or not cache.exists() or cache.stat().st_size == 0:
                continue
            n_baixados += 1
            time.sleep(0.2)
        texto = normalizar_pdf_text(pncp_pdfs.extrair_texto(cache))
        chars += len(texto)
        marc = detectar_marcadores(texto)
        for c in COLS_PRESENCA:
            if marc.get(c):
                marc_uni[c] = True
        tid = d.get('tipoDocumentoId')
        tipos.append(MAPA_TIPO_DOCUMENTO.get(tid, d.get('tipoDocumentoNome', '?')))

    n_mk = sum(1 for v in marc_uni.values() if v)
    reg['n_pdfs_processados'] = len(tipos)
    reg['tipos_pdf'] = ' | '.join(str(t) for t in tipos)
    reg['chars_total'] = chars
    reg['mk_score_engenharia'] = n_mk
    for c, v in marc_uni.items():
        reg[c] = bool(v)

    if chars < CHARS_MIN_LEGIVEL:
        n_sem_pdf_legivel += 1
        reg['classificacao_rito'] = 'indeterminado_sem_pdf_legivel'
    elif n_mk >= 2:
        reg['seguiu_rito'] = True
        reg['classificacao_rito'] = 'rotulacao_incorreta_processo_ok'
    else:
        reg['classificacao_rito'] = 'subenquadramento_real'
    resultados_rito.append(reg)

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(base_rito)} | baixados={n_baixados} cache={n_cache} '
              f'sem-doc={n_sem_doc} erro-api={n_erro} '
              f'sem-pdf-legível={n_sem_pdf_legivel}')

df_rito = pd.DataFrame(resultados_rito)
n_lidos = int((df_rito['chars_total'] >= CHARS_MIN_LEGIVEL).sum()) if len(df_rito) else 0
print(f'\n{len(df_rito)} suspeitos analisados | PDFs baixados={n_baixados}, '
      f'cache={n_cache}, lidos com sucesso={n_lidos}')
print('\nDistribuição do veredito de rito:')
print(df_rito['classificacao_rito'].value_counts().to_string())
if n_lidos == 0:
    print('\n⚠ Nenhum PDF legível. Veja o DIAGNÓSTICO acima:')
    print('  - se status="erro_NNN" → erro HTTP da API (talvez 404)')
    print('  - se docs=0 mesmo com fallback → o NCP não tem documento '
          'expostoss na API de integração; é um limite estrutural do PNCP.')
    print('  - se docs>0 mas baixados=0 → o nome do campo do sequencial '
          'mudou; me mande a saída de "campos do 1º doc".')

### 12.3 Veredito jurídico — distribuição + top violações

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

cont = df_rito['classificacao_rito'].value_counts()
cores_rito = {
    'subenquadramento_real':            '#d62728',   # vermelho — provável violação
    'rotulacao_incorreta_processo_ok':  '#ff7f0e',   # laranja — só erro cadastral
    'indeterminado_sem_pdf_legivel':    '#9aa0a6',
    'indeterminado_sem_documento':      '#c7c7c7',
    'indeterminado_erro_api':           '#e0e0e0',
}
axes[0].barh(cont.index.astype(str)[::-1], cont.values[::-1],
             color=[cores_rito.get(c, '#1f77b4') for c in cont.index[::-1]])
for k, v in enumerate(cont.values[::-1]):
    axes[0].text(v, k, f' {v}', va='center')
axes[0].set_title('Veredito de rito dos suspeitos confirmados')
axes[0].set_xlabel('nº de contratos')

com_doc = df_rito[df_rito['chars_total'] >= 300]
if len(com_doc):
    axes[1].hist(com_doc['mk_score_engenharia'], bins=range(0, 10),
                  color='#1f77b4', edgecolor='white', align='left')
    axes[1].axvline(2, color='#d62728', ls='--', lw=1, label='limiar rito (2)')
    axes[1].set_title('Marcadores de engenharia nos PDFs legíveis')
    axes[1].set_xlabel('nº de categorias presentes'); axes[1].set_xticks(range(0, 9))
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Nenhum PDF legível baixado\n(contratos não expõem TR/PB)',
                  ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_axis_off()
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/rito_distribuicao.png', dpi=120, bbox_inches='tight')
plt.show()

# Casos de PROVÁVEL VIOLAÇÃO: confirmados pelo LLM + PDF legível + sem rito
subenq_reais = df_rito[df_rito['classificacao_rito'] == 'subenquadramento_real']
print(f'\n⚠ {len(subenq_reais)} PROVÁVEIS VIOLAÇÕES da Lei 14.133/2021 '
      f'(objeto de engenharia confirmado + PDF legível SEM marcadores de rito):\n')
if len(subenq_reais):
    display_cols = ['numeroControlePNCP', 'objeto', 'mk_score_engenharia', 'tipos_pdf']
    display(subenq_reais[display_cols].head(20))
else:
    print('(nenhum — ou os PDFs não puderam ser lidos; ver indeterminados)')

### 12.4 Exporta CSV + atualiza o relatório .md

In [ ]:
# Salva o CSV do rito. O relatório .md consolidado é gerado na seção 14
# (que já inclui esta análise) — aqui só persistimos a tabela.
caminho_csv_rito = f'{PASTA_RESULTADOS}/11_analise_rito.csv'
df_rito.to_csv(caminho_csv_rito, index=False, encoding='utf-8-sig')
print(f'CSV salvo: {caminho_csv_rito} ({len(df_rito)} linhas)')

dist = df_rito['classificacao_rito'].value_counts().to_dict()
print('\nResumo do rito:')
for k, v in dist.items():
    print(f'  {k}: {v}')
n_viol = dist.get('subenquadramento_real', 0)
n_ok = dist.get('rotulacao_incorreta_processo_ok', 0)
n_indet = sum(v for k, v in dist.items() if k.startswith('indeterminado'))
print(f'\n→ {n_viol} prováveis violações | {n_ok} rotulação incorreta (rito ok) '
      f'| {n_indet} indeterminados (sem PDF legível)')
print('\nProssiga para a seção 14 (exportação consolidada) para gerar o '
      'relatório .md final com tudo.')

## 13. Agente LLM com ferramentas (sem LangChain)

Agente minimalista: o Gemini decide qual ferramenta chamar respondendo em JSON,
executamos e repetimos até ele concluir.

**Ferramentas:** `busca_semantica_grafo(query)` e `listar_top_suspeitos(n)`.

Sem LangChain — a API muda entre versões e quebra (o erro `ImportError` do
`AgentExecutor`). Depende só de `google.generativeai` + `json` + `re`.

In [ ]:
"""
Agente LLM minimalista — sem LangChain.

Loop manual: prompt → Gemini → parse JSON → executa tool ou responde.
Estável porque depende só de google.generativeai + json/regex (vs. LangChain
que muda a API entre versões e quebra com frequência).
"""
import json, re

def busca_semantica_grafo(query=''):
    if not query:
        return 'Por favor, forneça uma consulta.'
    q_emb = sbert.encode([query], normalize_embeddings=True)[0]
    sims = cosine_similarity([q_emb], X_emb)[0]
    idx = int(np.argmax(sims))
    viz = list(G_emb.neighbors(idx)) + [idx]
    textos = []
    for v in viz[:5]:
        if v < len(df_amostra):
            row = df_amostra.iloc[v]
            textos.append(f"[{row.get('rotulo', '?')}] "
                           f"{row.get('numeroControlePNCP', '?')}: "
                           f"{str(row['text'])[:300]}")
    return (f'Contratos similares a "{query}":\n\n'
            + '\n\n===\n\n'.join(textos)
            + '\n\n=== Fim ===')

def listar_top_suspeitos(n=10):
    """Top-N do ranking final consolidado (3 sinais) da seção 11."""
    if 'df_ranking' not in globals():
        return '(rode a seção 11 primeiro — falta o df_ranking)'
    sub = df_ranking.head(int(n))
    return '\n'.join(
        f"- [{r['n_sinais']} sinais] {r['numeroControlePNCP']}: "
        f"{str(r['text'])[:120]}"
        for _, r in sub.iterrows()
    )

_TOOLS = {
    'busca_semantica_grafo': busca_semantica_grafo,
    'listar_top_suspeitos': listar_top_suspeitos,
}

SYSTEM_AGENTE = """Você é um agente analítico para auditoria de contratos
públicos PNCP. Tem acesso a estas ferramentas:

- busca_semantica_grafo(query: str): busca contratos similares a um texto.
- listar_top_suspeitos(n: int): lista os top-N suspeitos do ranking final.

Quando precisar de dados, responda APENAS no JSON:
{"acao": "<nome_ferramenta>", "argumentos": {...}}

Quando tiver dados suficientes, responda APENAS no JSON:
{"acao": "responder", "resposta": "<texto natural com a conclusão>"}

Não escreva nada fora do JSON."""

def _parse_acao(txt):
    if not txt:
        return None
    matches = re.findall(r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", txt, flags=re.DOTALL)
    for m in reversed(matches):
        try:
            return json.loads(m)
        except Exception:
            continue
    return None

def agente(pergunta, max_passos=4, modelo=None):
    """Loop: LLM → tool → LLM → ... → resposta."""
    modelo = modelo or MODELO_LLM
    contexto = f"Pergunta: {pergunta}"
    for passo in range(max_passos):
        resp_txt = llm_task(modelo, SYSTEM_AGENTE, contexto)
        parsed = _parse_acao(resp_txt) or {}
        acao = parsed.get('acao')
        if acao == 'responder':
            return parsed.get('resposta', '(sem resposta)')
        if acao in _TOOLS:
            args = parsed.get('argumentos', {})
            try:
                saida = _TOOLS[acao](**args)
            except Exception as e:
                saida = f'(erro na ferramenta: {e})'
            contexto += f'\n\nUsei {acao}({args}).\nResultado:\n{saida}'
            print(f'[passo {passo+1}] {acao}({args})')
        else:
            # Ação desconhecida — força uma resposta final direta.
            final = llm_task(modelo,
                'Responda à pergunta do usuário em português, de forma direta, '
                'usando o que já foi coletado no contexto. Não use JSON.',
                contexto)
            return final or '(sem resposta)'
    # Esgotou os passos — pede uma síntese final.
    final = llm_task(modelo,
        'Com base no contexto coletado, responda à pergunta do usuário em '
        'português, de forma direta e objetiva. Não use JSON.', contexto)
    return final or '(agente atingiu limite de passos)'

In [ ]:
resposta = agente(
    'Liste os principais contratos suspeitos de subenquadramento '
    '(use n=30 ou mais na ferramenta) e descreva os padrões comuns entre eles.'
)
print('\n--- RESPOSTA FINAL ---\n')
print(resposta)

## 14. Exportação consolidada (.md + CSVs + gráficos)

Salva tabelas (CSV), gráficos (HTML/PNG) e um relatório `.md` em
`resultados_analise_textual/` — pasta **separada** dos resultados do pipeline.

In [ ]:
import datetime as _dt

def _salvar_csv(nome, df):
    if df is not None and len(df):
        df.to_csv(f'{PASTA_RESULTADOS}/{nome}.csv', index=False,
                  encoding='utf-8-sig')
        return f'- `{nome}.csv` — {len(df):,} linhas'
    return f'- `{nome}.csv` — (vazio)'

def _g(nome):
    return globals().get(nome)

linhas_csv = []
for nome, var in [('01_top_termos_tfidf', 'df_word_tfidfs'),
                  ('02_top_bigramas', 'df_bi'),
                  ('03_top_trigramas', 'df_tri'),
                  ('04_descritores_cluster_tfidf', 'df_descriptors'),
                  ('05_indicadores_cluster_llm', 'df_indicadores'),
                  ('07_voto_knn_geral', 'df_votos'),
                  ('08_ranking_suspeitos', 'df_ranking'),
                  ('09_veredito_llm', 'df_veredictos'),
                  ('11_analise_rito', 'df_rito')]:
    linhas_csv.append(_salvar_csv(nome, _g(var)))
if 'clusters_suspeitos' in globals():
    linhas_csv.append(_salvar_csv('06_clusters_suspeitos',
                                   clusters_suspeitos.reset_index()))

# ── Impacto financeiro + órgãos ─────────────────────────────────────────────
analise_valor = ''
if 'df_ranking' in globals() and 'valor' in df_ranking.columns:
    fortes = df_ranking[df_ranking['n_sinais'] >= 2].copy()
    fortes['valor'] = pd.to_numeric(fortes['valor'], errors='coerce')
    valor_total = fortes['valor'].sum()
    col_org = 'razaoSocialOrgao' if 'razaoSocialOrgao' in fortes.columns else None
    top_org_txt = ''
    if col_org:
        top_org = (fortes.groupby(col_org)
                   .agg(n=('numeroControlePNCP', 'count'), valor=('valor', 'sum'))
                   .sort_values('n', ascending=False).head(10))
        top_org.reset_index().to_csv(
            f'{PASTA_RESULTADOS}/10_top_orgaos_suspeitos.csv',
            index=False, encoding='utf-8-sig')
        linhas_csv.append(f'- `10_top_orgaos_suspeitos.csv` — {len(top_org)} órgãos')
        ll = ['| Órgão | nº | valor (R$) |', '|---|---|---|']
        for org, r in top_org.iterrows():
            v = r['valor'] if pd.notna(r['valor']) else 0
            ll.append(f"| {str(org)[:45]} | {int(r['n'])} | {v:,.0f} |")
        top_org_txt = '\n'.join(ll)
    analise_valor = (f"\n## Impacto financeiro (candidatos com >=2 sinais)\n\n"
                     f"- Contratos: **{len(fortes):,}**\n"
                     f"- Valor agregado: **R$ {valor_total:,.2f}**\n\n"
                     f"### Top 10 órgãos por nº de suspeitos\n\n{top_org_txt}\n")

# ── Veredito LLM com justificativa completa ─────────────────────────────────
veredito_txt = ''
if 'df_veredictos' in globals() and len(df_veredictos):
    conf = df_veredictos[df_veredictos['confirmado_subenq']]
    ll = ['| NCP | objeto | classe | conf | justificativa |',
          '|---|---|---|---|---|']
    for _, r in df_veredictos.iterrows():
        obj = str(r['objeto'])[:55].replace('|', '/')
        mot = str(r['llm_motivo'])[:110].replace('|', '/')
        ll.append(f"| {r['numeroControlePNCP']} | {obj} | {r['llm_classe']} | "
                   f"{r['llm_confianca']:.1f} | {mot} |")
    veredito_txt = (f"\n## Veredito do LLM ({len(conf)}/{len(df_veredictos)} "
                    f"confirmados como engenharia/obras)\n\n" + '\n'.join(ll) + '\n')

# ── Análise de rito ─────────────────────────────────────────────────────────
rito_txt = ''
if 'df_rito' in globals() and len(df_rito):
    dist = df_rito['classificacao_rito'].value_counts().to_dict()
    n_viol = dist.get('subenquadramento_real', 0)
    n_ok = dist.get('rotulacao_incorreta_processo_ok', 0)
    n_indet = sum(v for k, v in dist.items() if k.startswith('indeterminado'))
    n_lidos = int((df_rito['chars_total'] >= 300).sum())
    subq = df_rito[df_rito['classificacao_rito'] == 'subenquadramento_real']
    ll = ['| NCP | objeto | mk_score | PDFs |', '|---|---|---|---|']
    for _, r in subq.head(15).iterrows():
        obj = str(r['objeto'])[:55].replace('|', '/')
        ll.append(f"| {r['numeroControlePNCP']} | {obj} | "
                   f"{int(r['mk_score_engenharia'])} | {r.get('tipos_pdf', '')[:30]} |")
    tabela_viol = '\n'.join(ll) if len(subq) else '(nenhuma confirmada com PDF legível)'
    rito_txt = f"""
## Análise de rito de engenharia (seção 12)

Para os suspeitos confirmados pelo LLM, baixamos os PDFs anexados no PNCP e
buscamos 8 marcadores legais (ART, CREA, Engenheiro Resp., ATC, Projeto
Básico, Obra/Serv. Eng., ABNT, Lei 14.133). Critério: ≥2 categorias = rito
seguido.

- PDFs lidos com sucesso: **{n_lidos}** de {len(df_rito)}
- `subenquadramento_real` (PDF legível SEM rito) — **{n_viol}** → provável
  violação da Lei 14.133/2021
- `rotulacao_incorreta_processo_ok` (rito seguido) — {n_ok}
- `indeterminado` (sem PDF legível) — {n_indet}

> Limite conhecido: contratos (tipo 2 no NCP) em geral não expõem TR/Projeto
> Básico pela API de integração — o TR/PB pertence à COMPRA, não ao contrato.
> Por isso muitos ficam "indeterminado". Para conclusão definitiva, casar o
> contrato com sua compra de origem (trabalho futuro).

### Prováveis violações com PDF legível

{tabela_viol}
"""

# ── Números-chave ───────────────────────────────────────────────────────────
n_amostra = len(df_amostra)
dist_rotulo = df_amostra['rotulo'].value_counts().to_dict()
n_geral = int((df_amostra['rotulo'] == 'geral').sum())
n_2 = int((df_ranking['n_sinais'] >= 2).sum()) if 'df_ranking' in globals() else 0
n_3 = int((df_ranking['n_sinais'] >= 3).sum()) if 'df_ranking' in globals() else 0
n_conf = int(df_veredictos['confirmado_subenq'].sum()) if 'df_veredictos' in globals() else 0
n_ver = len(df_veredictos) if 'df_veredictos' in globals() else 0

relatorio = f"""# Resultados — Análise Textual de Contratos PNCP

_Gerado em {_dt.datetime.now():%Y-%m-%d %H:%M}_

## Objetivo

Identificar contratos rotulados como **serviços gerais** que na verdade são
**engenharia/obras** (subenquadramento, Lei 14.133/2021), usando os rótulos
`engenharia`/`obras` como referência (PU learning). Apenas engenharia
(CREA/ART) e obras — arquitetura (CAU/RRT) fora.

## Base analisada

- Amostra: **{n_amostra:,}** contratos | {', '.join(f'{k}={v}' for k, v in dist_rotulo.items())}
- Contratos `geral` investigados: **{n_geral:,}**

## Pipeline de detecção

1. **Pré-filtragem (seção 11):** 4 sinais — classificador supervisionado
   P(eng/obras), voto k-NN, pureza de cluster, regex de engenharia óbvia.
2. **Validação semântica (LLM):** veredito de classe correta com regra de ouro
   (locação/aquisição/evento = geral).
3. **Rito (seção 12):** marcadores legais nos PDFs → violação vs. erro cadastral.

## Resultados

- Candidatos com **>=2 sinais**: **{n_2}** | **>=3 sinais**: **{n_3}**
- Top-{n_ver} validados pelo LLM → **{n_conf}** confirmados engenharia/obras
{analise_valor}{veredito_txt}{rito_txt}
## Arquivos gerados

### Tabelas (CSV)
{chr(10).join(linhas_csv)}

### Gráficos
- `grafo_tfidf_cluster.html`, `grafo_sbert_rotulo.html`, `grafo_regularizacao.html`
- `sinais_continuos.png`, `sinais_subenquadramento.png`, `veredito_llm.png`
- `rito_distribuicao.png`

## Conclusão

O estudo identifica contratos rotulados "geral" que são de engenharia/obras
(rotulação incorreta) e, entre os com PDF legível, separa os que **não
seguiram o rito** (provável violação da Lei 14.133/2021) dos que seguiram
(apenas erro de cadastro). Casar contrato↔compra para recuperar TR/PB dos
indeterminados é o passo seguinte.
"""

caminho_md = f'{PASTA_RESULTADOS}/relatorio_analise_textual.md'
with open(caminho_md, 'w', encoding='utf-8') as f:
    f.write(relatorio)
print(f'Relatório consolidado salvo em:\n  {caminho_md}\n')
print(relatorio)